In [2]:
import math

In [3]:
R_EARTH_KM = 6378.137
GM = 3.0986004418e5
GEO_ORBIT_RADIUS_KM = 42164.17
SPEED_OF_LIGHT = 299_792_458

In [4]:
satellite_altiude = 350
orbit_radius = R_EARTH_KM + satellite_altiude
angular_velocity = (GM / orbit_radius**3)**0.5
print(f"Orbit radius: {orbit_radius} km")
print(f"Angular velocity of satellite: {angular_velocity} rad/s")

Orbit radius: 6728.137 km
Angular velocity of satellite: 0.001008650132925822 rad/s


In [5]:
orbital_period = 2 * math.pi / angular_velocity
orbital_period_minutes = orbital_period / 60
print(f"Orbital period: {orbital_period_minutes} min")

Orbital period: 103.82168375459983 min


In [6]:
orbital_velocity = angular_velocity * orbit_radius
print(f"Orbital velocity: {orbital_velocity} km/s")

Orbital velocity: 6.786336279393142 km/s


In [7]:
orbital_radius_apogee = R_EARTH_KM + 5000
orbital_radius_perigee = R_EARTH_KM + 800
eccentricity = (orbital_radius_apogee - orbital_radius_perigee) /(orbital_radius_apogee + orbital_radius_perigee)
print(f"Apogee radius: {orbital_radius_apogee} km")
print(f"Perigee radius: {orbital_radius_perigee} km")
print(f"Eccentricity: {eccentricity}")

Apogee radius: 11378.136999999999 km
Perigee radius: 7178.137 km
Eccentricity: 0.22633854188615665


In [8]:
GEO_TO_EARTH_RATIO = GEO_ORBIT_RADIUS_KM / R_EARTH_KM
print(f"Earth radius to GEO orbit radius ratio: {GEO_TO_EARTH_RATIO}")

Earth radius to GEO orbit radius ratio: 6.610734451141454


In [9]:

def calculate_central_angle(sat_pos, gs_pos):
    
    sat_lon_rad = math.radians(sat_pos[1])
    gs_lat_rad = math.radians(gs_pos[0])
    gs_lon_rad = math.radians(gs_pos[1])

    return math.degrees(math.acos(
        math.cos(gs_lat_rad) * math.cos(sat_lon_rad - gs_lon_rad)
    ))
    
def calculate_elevation_angle(sat_pos, gs_pos):
    central_angle = calculate_central_angle(sat_pos, gs_pos)
    return math.degrees(math.atan(
        (GEO_TO_EARTH_RATIO - math.cos(math.radians(central_angle))) / 
        math.sin(math.radians(central_angle))) - math.radians(central_angle)
    )
    
def calculate_intermediate_angle(sat_pos, gs_pos):
    
    lon_diff_rad = abs(math.radians(sat_pos[1]) - math.radians(gs_pos[1]))
    el_rad = math.atan(math.tan(lon_diff_rad) / abs(math.sin(math.radians(gs_pos[0]))))
    
    return math.degrees(el_rad)

def calculate_azimuth_angle(sat_pos, gs_pos):
    
    intermediate_angle = calculate_intermediate_angle(sat_pos, gs_pos)
    
    if sat_pos[1] == gs_pos[1]:
        if sat_pos[0] > gs_pos[0]:
            return 0
        else:
            return 180
    
    # Ground station is in the northern hemisphere
    if gs_pos[0] > 0:
        # Satellite is SE of ground station
        if sat_pos[0] < gs_pos[0] and sat_pos[1] > gs_pos[1]:
            return 180 - intermediate_angle
        # Satellite is SW of ground station
        elif sat_pos[0] < gs_pos[0] and sat_pos[1] < gs_pos[1]:
            return 180 + intermediate_angle
        else:
            raise ValueError
    # Ground station is in the southern hemisphere
    else:
        # Satellite is NE of ground station
        if sat_pos[0] > gs_pos[0] and sat_pos[1] > gs_pos[1]:
            return intermediate_angle
        # Satellite is NW of ground station
        elif sat_pos[0] > gs_pos[0] and sat_pos[1] < gs_pos[1]:
            return 360 - intermediate_angle
        else:
            raise ValueError


In [10]:
# From Example 2.6 as a sanity check
satellite_position = (0, 66)
gs_position = (52, 0)
central_angle = calculate_central_angle(satellite_position, gs_position)
print(f"Central angle: {central_angle} degrees")
print(f"Elevation angle: {calculate_elevation_angle(satellite_position, gs_position)} degrees")
print(f"Intermediate angle: {calculate_intermediate_angle(satellite_position, gs_position)} degrees")
print(f"Azimuth angle: {calculate_azimuth_angle(satellite_position, gs_position)} degrees")

Central angle: 75.49810160755791 degrees
Elevation angle: 5.847030478244452 degrees
Intermediate angle: 70.66683422234857 degrees
Azimuth angle: 109.33316577765143 degrees


In [11]:
satellite_position = (0, 109)
gs_position = (37.22, 80.42)
print(f"Elevation angle: {calculate_elevation_angle(satellite_position, gs_position)} degrees")
print(f"Azimuth angle: {calculate_azimuth_angle(satellite_position, gs_position)} degrees")

Elevation angle: 37.47485556137115 degrees
Azimuth angle: 137.99313995691307 degrees


In [12]:
satellite_position = (0, 109)
gs_position = (46, 109)
print(f"Elevation angle: {calculate_elevation_angle(satellite_position, gs_position)} degrees")
print(f"Azimuth angle: {calculate_azimuth_angle(satellite_position, gs_position)} degrees")

Elevation angle: 37.067397260501295 degrees
Azimuth angle: 180 degrees


In [13]:
satellite_position = (0, 109)
gs_position = (34, 118)
print(f"Elevation angle: {calculate_elevation_angle(satellite_position, gs_position)} degrees")
print(f"Azimuth angle: {calculate_azimuth_angle(satellite_position, gs_position)} degrees")

Elevation angle: 49.30782252603563 degrees
Azimuth angle: 195.81411483599757 degrees


In [14]:
satellite_position = (0, 30)
station_1 = (-22.91, 43.17)
station_2 = (-33.45, 70.67)
print(f"Elevation angle: {calculate_elevation_angle(satellite_position, station_1)} degrees")
print(f"Azimuth angle: {calculate_azimuth_angle(satellite_position, station_1)} degrees")
print(f"Elevation angle: {calculate_elevation_angle(satellite_position, station_2)} degrees")
print(f"Azimuth angle: {calculate_azimuth_angle(satellite_position, station_2)} degrees")

Elevation angle: 59.32642752480712 degrees
Azimuth angle: 328.99030209829294 degrees
Elevation angle: 31.88043568733296 degrees
Azimuth angle: 302.68093484007716 degrees


In [15]:
8/(1.36 * 10)
7.2/(1.36 * 10)
print(f"Efficiency at BOL: {8/(1.36 * 10)}")
print(f"Efficiency at EOL: {7.2/(1.36 * 10)}")

Efficiency at BOL: 0.588235294117647
Efficiency at EOL: 0.5294117647058824


In [16]:
power = 2.5 + 2.5*1.25
print(f"Power required: {power} kW")

Power required: 5.625 kW


In [17]:
current = power*1000/50
print(f"Current required: {current} A")

Current required: 112.5 A


In [18]:
hours = 46/60
ampere_hours = current * hours
print(f"Total ampere-hours required: {ampere_hours} Ah")

Total ampere-hours required: 86.25 Ah


In [19]:
wavelength = SPEED_OF_LIGHT / 12.2e9
print(f"Wavelength of 12.2 GHz signal: {wavelength} m")

Wavelength of 12.2 GHz signal: 0.024573152295081967 m


In [26]:
area = math.pi*0.3**2
gain = 0.70*4 * math.pi * area / wavelength**2
gain_db = 10 * math.log10(gain)
print(f"Area of the antenna: {area} m^2")
print(f"Gain of the antenna: {gain}")
print(f"Gain of the antenna in dB: {gain_db}")

Area of the antenna: 0.2827433388230814 m^2
Gain of the antenna: 4118.874331527527
Gain of the antenna in dB: 36.147785416634534


In [21]:
beamwidth = 70*wavelength/0.6
print(f"Beamwidth of the antenna: {beamwidth} degrees")

Beamwidth of the antenna: 2.866867767759563 degrees


In [25]:
gain = 33000/beamwidth**2
print(f"Gain of the antenna: {gain}")
gain_db = 10 * math.log10(gain)
print(f"Gain of the antenna in dB: {gain_db}")

Gain of the antenna: 4015.1207999899675
Gain of the antenna in dB: 36.03698616110302
